In [ ]:
import os
import time
import csv
import requests
from google.colab import drive

# 1. MOUNT GOOGLE DRIVE & KONFIGURASI
print("Meminta akses Google Drive...")
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/GrafikaKomputer/Final_Project'
IMAGE_DIR = os.path.join(BASE_DIR, 'images')
CSV_FILE = os.path.join(BASE_DIR, 'metadata_artefak.csv')

os.makedirs(IMAGE_DIR, exist_ok=True)

TARGET_COUNT = 150
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# 2. FUNGSI PENCARIAN API THE MET
def get_indonesian_artifacts():
    """Mengambil daftar Object ID dari The Met API dengan keyword Indonesia."""
    print("Menghubungi server The Met API untuk mencari artefak Indonesia...")
    url = "https://collectionapi.metmuseum.org/public/collection/v1/search?hasImages=true&q=Indonesia"

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        data = response.json()

        if data and data.get('objectIDs'):
            object_ids = data['objectIDs']
            print(f"Ditemukan {len(object_ids)} total ID artefak potensial dari pencarian awal.")
            return object_ids
        else:
            print("Pencarian tidak menemukan ID objek. Periksa koneksi atau parameter API.")
            return []
    except Exception as e:
        print(f"Error fatal saat mencari daftar artefak: {e}")
        return []

# 3. PIPELINE UNDUHAN & EKSTRAKSI METADATA
def execute_pipeline(artifact_ids):
    if not artifact_ids:
        return

    downloaded_count = 0
    file_exists = os.path.isfile(CSV_FILE)

    with open(CSV_FILE, mode='a', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        if not file_exists:
            writer.writerow(['Object_ID', 'Judul', 'Budaya', 'Lisensi', 'URL_Gambar_API', 'URL_Web_Sumber'])

        print(f"\nMemulai proses filter Open Access dan unduhan target: {TARGET_COUNT} gambar...")

        for obj_id in artifact_ids:
            if downloaded_count >= TARGET_COUNT:
                print(f"\n=== TARGET {TARGET_COUNT} GAMBAR TERCAPAI ===")
                break

            obj_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{obj_id}"

            try:
                # 1. Ambil Metadata Objek
                res = requests.get(obj_url, headers=HEADERS, timeout=10)
                res.raise_for_status()
                obj_data = res.json()

                # FILTER KERAS: Harus CC0 (Public Domain) dan punya gambar utama
                if not obj_data.get('isPublicDomain') or not obj_data.get('primaryImage'):
                    time.sleep(0.5) # Jeda ringan untuk API cache miss
                    continue

                image_url = obj_data['primaryImage']
                title = obj_data.get('title', 'Tanpa Judul').replace('\r', '').replace('\n', ' ')
                culture = obj_data.get('culture', 'Budaya Tidak Diketahui')
                source_url = obj_data.get('objectURL', '')

                ext = image_url.split('.')[-1].lower()
                if ext not in ['jpg', 'jpeg', 'png']:
                    ext = 'jpg'

                filename = f"artefak_{obj_id}.{ext}"
                filepath = os.path.join(IMAGE_DIR, filename)

                if os.path.exists(filepath):
                    print(f"[-] Melewati ID {obj_id} - File sudah ada di Drive.")
                    downloaded_count += 1
                    continue

                # 2. Unduh File Gambar
                print(f"Mengunduh [{downloaded_count + 1}/{TARGET_COUNT}]: {title[:40]}... (ID: {obj_id})")
                img_res = requests.get(image_url, headers=HEADERS, timeout=20)
                img_res.raise_for_status()

                # 3. Tulis ke Google Drive
                with open(filepath, 'wb') as f_img:
                    f_img.write(img_res.content)

                # 4. Tulis Metadata ke CSV
                writer.writerow([obj_id, title, culture, 'CC0 (Public Domain)', image_url, source_url])

                downloaded_count += 1

            except requests.exceptions.RequestException as req_err:
                print(f"[!] Jaringan terputus saat memproses ID {obj_id}: {req_err}")
            except Exception as e:
                print(f"[!] Gagal mengeksekusi ID {obj_id}: {e}")

            # RATE LIMIT WAJIB: Jangan sampai IP Colab diblokir server The Met
            time.sleep(1.5)

# EKSEKUSI UTAMA
if __name__ == '__main__':
    ids_to_process = get_indonesian_artifacts()
    execute_pipeline(ids_to_process)
    print(f"\n[SELESAI] Silakan periksa folder Google Drive kamu di: {BASE_DIR}")

Meminta akses Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Menghubungi server The Met API untuk mencari artefak Indonesia...
Ditemukan 2666 total ID artefak potensial dari pencarian awal.

Memulai proses filter Open Access dan unduhan target: 150 gambar...
Mengunduh [1/150]: Ceremonial Object in the Shape of an Ax... (ID: 37743)
Mengunduh [2/150]: Head of a Buddha... (ID: 38267)
Mengunduh [3/150]: Box for Betel Leaves... (ID: 37747)
Mengunduh [4/150]: Hanging Lamp in the Form of a Kinnari... (ID: 39076)
Mengunduh [5/150]: The Transcendent Buddha Vairochana (?) S... (ID: 39021)
Mengunduh [6/150]: Seated Transcendent Buddha Vairochana... (ID: 39020)
Mengunduh [7/150]: Enthroned Buddha Attended by the Bodhisa... (ID: 72498)
Mengunduh [8/150]: Palepai maju (ceremonial banner)... (ID: 316457)
Mengunduh [9/150]: Buddha Attended by Bodhisattvas Avalokit... (ID: 38994)
Mengunduh [10/150]: Ax Hea